# National trends in peak annual streamflow

## Introduction

This notebook demonstrates a slightly more advanced application of the `dataretrieval.waterdata` module: assembling a dataset of historical annual peak streamflow and regressing peak discharge against time to look for trends — not at a single station, but across many.

## Setup
Before we begin any analysis, we'll need to set up our environment by importing a few modules.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

from dataretrieval import waterdata

## Basic usage
The `waterdata` module is the recommended interface to USGS water data and replaces the deprecated `nwis` module. Annual peak streamflow is retrieved with `waterdata.get_peaks()`, which returns a `pandas` data frame and a metadata object. To get started we need a monitoring location ID, a parameter code, and (optionally) a time window.

In [ ]:
# download annual peaks (discharge, parameter 00060) from a single site
df, md = waterdata.get_peaks(
    monitoring_location_id="USGS-03339000",
    parameter_code="00060",
    time="1970-01-01/..",
)
df.head()

All we require for the trend analysis are the peak date (`time`), the monitoring location ID (`monitoring_location_id`), and the peak streamflow (`value`). The Water Data API returns a flat (single-index) data frame with one row per annual peak.

## Preparing the regression
Next we'll define a function that applies ordinary least squares to peak discharge versus time. After grouping the dataset by `monitoring_location_id`, we apply the regression per monitoring location. Each location's result is returned as a row with the slope, y-intercept, p value, and standard error of the regression.

In [ ]:
def peak_trend_regression(df):
    # convert peak dates to days since the first peak for the regression
    peak_date = pd.to_datetime(df["time"])
    peak_d = (peak_date - peak_date.min()) / np.timedelta64(1, "D")

    # normalize the peak discharge values
    value = (df["value"] - df["value"].mean()) / df["value"].std()

    slope, intercept, _r_value, p_value, std_error = stats.linregress(peak_d, value)

    return pd.Series(
        {
            "slope": slope,
            "intercept": intercept,
            "p_value": p_value,
            "std_error": std_error,
        }
    )

## Preparing the analysis

In [ ]:
def peak_trend_analysis(state_names, start_date):
    """
    state_names : list
        state names to include in the analysis, e.g. ["Illinois", "Indiana"].
    start_date : string
        the date to use as the beginning of the analysis (YYYY-MM-DD).
    """
    final_df = pd.DataFrame()

    for state in state_names:
        # find stream gages in the state
        sites, _ = waterdata.get_monitoring_locations(
            state_name=state, site_type_code="ST", skip_geometry=True
        )
        # download annual peak discharge for those sites
        df, _ = waterdata.get_peaks(
            monitoring_location_id=sites["monitoring_location_id"].tolist(),
            parameter_code="00060",
            time=f"{start_date}/..",
        )
        # group the data by site and apply our regression
        temp = (
            df.groupby("monitoring_location_id")
            .apply(peak_trend_regression)
            .dropna()
        )
        # drop any insignificant results
        temp = temp[temp["p_value"] < 0.05]

        # join site metadata (for mapping) with the trend results
        merged = pd.merge(
            sites, temp, right_index=True, left_on="monitoring_location_id"
        )
        final_df = pd.concat([final_df, merged], ignore_index=True)

    return final_df

Running the analysis for every state since 1970 would pull a large amount of data from the Water Data API and could burden the service. To keep this demo light, we run a single small state — Rhode Island — below.

In [ ]:
# Download peak discharge for every stream gage in Rhode Island and run the
# trend regression on each. For larger states the async chunker (default
# concurrency 16) automatically fans the ``get_peaks`` call across many sites;
# Rhode Island is small enough to return in a single request.
start = "1970-01-01"
states = ["Rhode Island"]
final_df = peak_trend_analysis(state_names=states, start_date=start)
final_df.head()

The cell above ran the full analysis for a single small state (Rhode Island) and returned `final_df` — one row per gage whose peak-discharge trend is statistically significant (`p_value < 0.05`), carrying the regression slope, intercept, p-value, and standard error.

`final_df` pairs each station's trend statistics with its monitoring-location metadata (`monitoring_location_id`, `state_name`, `site_type_code`, …). Because `peak_trend_analysis` requests the locations with `skip_geometry=True`, `final_df` carries no coordinate columns; drop that argument if you want the `geometry` for mapping.

## Plotting the results
The commented cell below sketches how one might map the results with `basemap` and `matplotlib`, coloring monitoring locations with increasing peak annual discharge in red and decreasing peaks in blue. It is left commented because `basemap` is awkward to install on a remote machine. Note that it refers to a pre-generated national result set using the legacy NWIS coordinate columns (`dec_lat_va` / `dec_long_va`) rather than the Rhode Island `final_df` computed above; to map `final_df` directly, rerun `peak_trend_analysis` without `skip_geometry=True` and use the resulting `geometry` column.

In [ ]:
# Currently commented out as there isn't an easy way to install mpl_toolkits
# on a remote machine without spinning up a full geospatial stack.

# from mpl_toolkits.basemap import Basemap, cm
# import matplotlib.pyplot as plt

# fig = plt.figure(num=None, figsize=(10, 6) )

# # setup a basemap covering the contiguous United States
# m = Basemap(width=5500000, height=4000000, resolution='l',
#             projection='aea', lat_1=36., lat_2=44, lon_0=-100, lat_0=40)


# # add coastlines
# m.drawcoastlines(linewidth=0.5)

# # add parallels and meridians.
# m.drawparallels(np.arange(-90.,91.,15.),labels=[True,True,False,False],dashes=[2,2])
# m.drawmeridians(np.arange(-180.,181.,15.),labels=[False,False,False,True],dashes=[2,2])

# # add boundaries and rivers
# m.drawcountries(linewidth=1, linestyle='solid', color='k' )
# m.drawstates(linewidth=0.5, linestyle='solid', color='k')
# m.drawrivers(linewidth=0.5, linestyle='solid', color='cornflowerblue')


# increasing = final_df[final_df['slope'] > 0]
# decreasing = final_df[final_df['slope'] < 0]

# #x,y = m(lons, lats)

# # categorical plots get a little  ugly in basemap
# m.scatter(increasing['dec_long_va'].tolist(),
#           increasing['dec_lat_va'].tolist(),
#           label='increasing', s=2, color='red',
#           latlon=True)

# m.scatter(decreasing['dec_long_va'].tolist(),
#           decreasing['dec_lat_va'].tolist(),
#           label='increasing', s=2, color='blue',
#           latlon=True)